# 006-F3 FIORA-OS 负离子小试（DEV run，not submission）

- 固定 OS checkpoint `fiora/resources/models/fiora_OS_v1.0.0.pt`（53,845,076 B，commit `e19ef82c9a6cb9dbac92bce23e914008f1aeb44e`，params 训练库 MSnLib v7）。代码 MIT；MSnLib v7 标注 CC BY 4.0，保留原始来源。不凭 `default` 之名选权重；NIST 相关一律排除。
- 支持 `[M+H]+/[M-H]-/[M]+/[M]-`，instrument 仅 HCD，CE 归一化尺度 100。`timsTOF→HCD` 是域迁移假设；按实际 eV 输入，NCE 不当 eV。
- 队列：34 个无 GL/ICE 证据 NP 分子中 19 个有 `[M-H]-` 谱为第一批；另 15 个仍不支持。加 3 个有 GL 支持正谱分子作域差对照（AIONOLUJZLIMTK 正负兼有、ADEBPBSSDYVVLD 纯正、ACWBQPMHZXGDFX 复杂）。候选固定 top50（窗口不足按实际，AAWZ 仅 14 等）；原预测 + FP 残差融合，不直接覆盖。
- 输入复用私有数据集 `ccjjllt/casmi26-glacier-np250` 内 `np250_gl_input.json`（mid/cands/spectra，无真值；融合在本地 CPU 做，本 notebook 只输出原始预测与相似度，不接触真值）。
- `enable_internet=true` 仅 DEV 下载（pinned 权重 + pinned 源码 pip）。最终提交保持离线；本轮跑通后记离线可行性（53MB 权重 + 源码可打包为数据集输入）。排行榜提交绝对不点。
- 止损：单格超 10 分钟无输出即停；RUNNING 超 60 分钟无新日志停下报告。push 最多两次。

In [ ]:
import sys, os, glob, json, time
print('python', sys.version)
print('INPUT_ROOT', sorted(os.listdir('/kaggle/input')))
cands = sorted(glob.glob('/kaggle/input/**/np250_gl_input.json', recursive=True))
print('np250_gl_input candidates', cands)
assert len(cands) >= 1, 'np250_gl_input.json not found under /kaggle/input: ' + str(sorted(os.listdir('/kaggle/input')))
PILOT_PATH = cands[0]
print('PILOT', PILOT_PATH, round(os.path.getsize(PILOT_PATH)/2**20, 2), 'MB')
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
except Exception as e:
    print('torch probe failed', repr(e)[:500])
    raise
try:
    import numpy
    print('numpy', numpy.__version__)
except Exception as e:
    print('numpy missing', repr(e)[:300])
    raise


In [ ]:
import os, json, time, hashlib, urllib.request
t0 = time.time()
COMMIT = 'e19ef82c9a6cb9dbac92bce23e914008f1aeb44e'
WURL = ('https://raw.githubusercontent.com/BAMeScience/fiora/' + COMMIT + '/fiora/resources/models/fiora_OS_v1.0.0.pt')
PURL = ('https://raw.githubusercontent.com/BAMeScience/fiora/' + COMMIT + '/fiora/resources/models/fiora_OS_v1.0.0_params.json')
WPATH = '/kaggle/working/fiora_OS_v1.0.0.pt'
PPATH = '/kaggle/working/fiora_OS_v1.0.0_params.json'
EXPECT_BYTES = 53845076
print('commit', COMMIT)
print('weight url', WURL)
print('bytes expect', EXPECT_BYTES)
# DEV-only internet download (pinned commit, no default drift, no NIST)
urllib.request.urlretrieve(WURL, WPATH)
urllib.request.urlretrieve(PURL, PPATH)
wb = os.path.getsize(WPATH)
print('downloaded weight bytes', wb, f'({time.time()-t0:.1f}s)')
assert wb == EXPECT_BYTES, f'weight size mismatch: got {wb}, expect {EXPECT_BYTES}'
h = hashlib.sha256()
with open(WPATH, 'rb') as f:
    for ch in iter(lambda: f.read(1<<20), b''):
        h.update(ch)
print('weight sha256', h.hexdigest())
params = json.load(open(PPATH))
print('params keys', sorted(params.keys()) if isinstance(params, dict) else type(params))
# params 可能是 {model_params: {...}} 或平铺；两种都兼容记录
mp = params.get('model_params', params) if isinstance(params, dict) else {}
print(json.dumps({k: mp.get(k) for k in ('version_number','training_library','gnn_type','depth','hidden_dimension','instrument','precursor_mode','rt_supported','ccs_supported')}, indent=1))
tl = str(mp.get('training_library',''))
assert 'MSnLib' in tl and 'v7' in tl, f'training_library not MSnLib v7: {tl!r}'
print('instrument field:', mp.get('instrument'))
print('precursor_mode field:', mp.get('precursor_mode'))
print('setup_features:', mp.get('setup_features'))
print('setup_categorical:', mp.get('setup_features_categorical_set'))
cat=str(mp.get('setup_features_categorical_set',''))
print('HCD_in_params:', ('HCD' in str(mp)))
assert str(mp.get('version_number',''))=='1.0.0', f"version not 1.0.0: {mp.get('version_number')}"
print('provenance ok: MSnLib v7 (CC BY 4.0 keep source) + MIT code; no NIST')
print(f'weight fetch done in {time.time()-t0:.1f}s')


In [ ]:
import subprocess, sys, os, json, time
t0 = time.time()
COMMIT = 'e19ef82c9a6cb9dbac92bce23e914008f1aeb44e'
assert os.path.getsize('/kaggle/working/fiora_OS_v1.0.0.pt') == 53845076
r = subprocess.run(['git', 'clone', 'https://github.com/BAMeScience/fiora', '/tmp/fiora'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-1500:])
assert os.path.isdir('/tmp/fiora/fiora'), 'clone failed'
r = subprocess.run(['git', '-C', '/tmp/fiora', 'checkout', COMMIT], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-800:])
r = subprocess.run(['git', '-C', '/tmp/fiora', 'rev-parse', 'HEAD'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
head = r.stdout.strip()
print('HEAD', head)
assert head == COMMIT, f'pinned commit mismatch: {head}'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '/tmp/fiora'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-2000:])
print(f'install done in {time.time()-t0:.1f}s')
r = subprocess.run(['fiora-predict', '--help'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[:2000])
assert 'INPUT' in r.stdout or '-i' in r.stdout, 'fiora-predict CLI not usable'
r = subprocess.run(['fiora-model-info', '-m', '/kaggle/working/fiora_OS_v1.0.0.pt', '--as-json'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[:2500])
print(f'fiora setup ok in {time.time()-t0:.1f}s')


In [ ]:
import json, os, csv, time
t0 = time.time()
M19 = ['AAWZDTNXLSGCEK','AMOGMTLMADGEOQ','APAJFZPFBHMFQR','BDCDNTVZSILEOY','CZMRCDWAGMRECN','FBPFZTCFMRRESA','FVYXIJYOAGAUQK','GRBXNADBNJGZRK','JDUWHZOLEDOQSR','QFJCIRLUMZQUOT','QIVBCDIJIAJPQS','QKXGRPHIBLICSC','QQTZGKRXYRFKIF','RNKMOGIPOMVCHO','RTIXKCRFFJGDFG','SEEGHKWOBVVBTQ','SQOJOAFXDQDRGF','UEDUENGHJMELGK','YHKYFYCSTJGMID']
CTRL = ['AIONOLUJZLIMTK','ADEBPBSSDYVVLD','ACWBQPMHZXGDFX']
TARGET = {}
for k in M19:
    TARGET[k] = '[M-H]-'
for k in CTRL:
    TARGET[k] = '[M+H]+'
import glob as _g0
_hits0 = sorted(_g0.glob('/kaggle/input/**/np250_gl_input.json', recursive=True))
assert _hits0, 'np250 input missing'
items = json.load(open(_hits0[0]))
by_mid = {x['mid']: x for x in items}
missing = [k for k in list(TARGET) if k not in by_mid]
assert not missing, f'target mids missing: {missing}'
SUPPORTED = {'[M+H]+','[M-H]-','[M]+','[M]-'}
pilot_meta = {'timsTOF_to_HCD': 'domain assumption (not QTOF mapping)', 'ce_semantics': 'eV only; NCE rejected', 'instrument_fixed': 'HCD', 'model': 'fiora-OS-v1.0.0-msnlib-v7', 'molecules': {}}
rows = []
n_unsup_spectra = 0
for mid, ad in TARGET.items():
    mol = by_mid[mid]
    sup = [s for s in mol['spectra'] if s.get('adduct') == ad and s.get('adduct') in SUPPORTED and s.get('ce_units') != 'NCE' and isinstance(s.get('ce_ev'), list) and len(s['ce_ev'])>0]
    unsup = len(mol['spectra']) - len([s for s in mol['spectra'] if s.get('adduct') == ad])
    n_unsup_spectra += sum(1 for s in mol['spectra'] if s.get('adduct') not in SUPPORTED or s.get('ce_units') == 'NCE')
    assert len(sup) >= 1, f'{mid}: no supported {ad} spectra (all adducts {[s.get("adduct") for s in mol["spectra"]]})'
    nce = [s for s in mol['spectra'] if s.get('ce_units') == 'NCE']
    assert not nce, f'{mid}: unexpected NCE spectra, must not cast to eV'
    flat = [float(v) for s in sup for v in s['ce_ev']]
    ce_mean = sum(flat)/len(flat)
    assert ce_mean == ce_mean and 0 < ce_mean <= 100, f'{mid}: bad ce_mean {ce_mean}'
    pilot_meta['molecules'][mid] = {'target_adduct': ad, 'n_cands': len(mol['cands']), 'n_spectra_total': len(mol['spectra']), 'n_supported_target': len(sup), 'ce_mean': ce_mean, 'ce_lists': [list(map(float, s['ce_ev'])) for s in sup], 'instruments_raw': sorted({str(s.get('instrument')) for s in sup})}
    for i, smi in enumerate(mol['cands']):
        assert isinstance(smi, str) and len(smi) >= 2, f'bad SMILES {mid} {i}'
        rows.append({'Name': f'{mid}__{i:03d}', 'mid': mid, 'SMILES': smi, 'Precursor_type': ad, 'CE': round(float(ce_mean), 2), 'Instrument_type': 'HCD'})
print('n_mols', len(pilot_meta['molecules']), 'n_rows', len(rows), 'unsupported_other_spectra', n_unsup_spectra)
assert len(rows) == sum(v['n_cands'] for v in pilot_meta['molecules'].values()), 'row count mismatch'
# 拆两块保证可观测性（单块黑盒超 10 分钟即停）
mids = list(TARGET.keys())
chunkA_mids = set(mids[:12])
for tag, sel in [('A', chunkA_mids), ('B', set(mids[12:]))]:
    sub = [r for r in rows if r['mid'] in sel]
    p = f'/kaggle/working/fiora_input_{tag}.csv'
    with open(p, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['Name','SMILES','Precursor_type','CE','Instrument_type'])
        w.writeheader()
        for r in sub:
            w.writerow({k: r[k] for k in ['Name','SMILES','Precursor_type','CE','Instrument_type']})
    print(tag, 'mids', len(sel), 'rows', len(sub), '->', p, round(os.path.getsize(p)/1024,1), 'KB')
with open('/kaggle/working/fiora_pilot_meta.json','w') as f:
    json.dump(pilot_meta, f, indent=1)
print(f'pilot build done in {time.time()-t0:.1f}s')


In [ ]:
import subprocess, os, time, glob
t0 = time.time()
W = '/kaggle/working/fiora_OS_v1.0.0.pt'
assert os.path.getsize(W) == 53845076
cmd = ['fiora-predict','-i','/kaggle/working/fiora_input_A.csv','-o','/kaggle/working/fiora_pred_A.mgf','--model',W,'--dev','cuda:0','--annotation','--no-rt','--no-ccs']
print(' '.join(cmd))
r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-3000:])
assert r.returncode == 0, f'fiora-predict A failed rc={r.returncode}'
assert os.path.exists('/kaggle/working/fiora_pred_A.mgf'), 'A.mgf missing'
print('A.mgf', round(os.path.getsize('/kaggle/working/fiora_pred_A.mgf')/1024,1), 'KB', f'in {time.time()-t0:.1f}s')
n_ions = sum(1 for ln in open('/kaggle/working/fiora_pred_A.mgf', errors='ignore') if ln.strip() == 'BEGIN IONS')
print('A BEGIN IONS', n_ions)


In [ ]:
import subprocess, os, time
t0 = time.time()
W = '/kaggle/working/fiora_OS_v1.0.0.pt'
cmd = ['fiora-predict','-i','/kaggle/working/fiora_input_B.csv','-o','/kaggle/working/fiora_pred_B.mgf','--model',W,'--dev','cuda:0','--annotation','--no-rt','--no-ccs']
print(' '.join(cmd))
r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-3000:])
assert r.returncode == 0, f'fiora-predict B failed rc={r.returncode}'
assert os.path.exists('/kaggle/working/fiora_pred_B.mgf'), 'B.mgf missing'
print('B.mgf', round(os.path.getsize('/kaggle/working/fiora_pred_B.mgf')/1024,1), 'KB', f'in {time.time()-t0:.1f}s')
n_ions = sum(1 for ln in open('/kaggle/working/fiora_pred_B.mgf', errors='ignore') if ln.strip() == 'BEGIN IONS')
print('B BEGIN IONS', n_ions)


In [ ]:
import json, os, csv, time, math
t0 = time.time()
import numpy as _np
TOL, PPM = 0.01, 20.0
def clean_peaks(mz, it, prec, floor=0.001, topk=512, max_over=2.0):
    mz=_np.asarray(list(mz),dtype=_np.float64); it=_np.asarray(list(it),dtype=_np.float64)
    if len(mz)==0: return mz.astype(_np.float32), it.astype(_np.float32)
    keep=(mz<=float(prec)+max_over)&(it>0)&(mz>0)
    mz,it=mz[keep],it[keep]
    if len(mz)==0: return mz.astype(_np.float32), it.astype(_np.float32)
    mx=float(it.max())
    keep=it>=0.001*mx; mz,it=mz[keep],it[keep]
    if len(mz)>topk:
        o=_np.argsort(it)[len(it)-topk:]; mz,it=mz[o],it[o]
    o=_np.argsort(mz); mz,it=mz[o],it[o]
    return mz.astype(_np.float32),(it/float(it.max())).astype(_np.float32)
def _entw(it,power=1.0):
    it=_np.asarray(it,dtype=_np.float64); p=_np.power(it,float(power)); s=float(p.sum())
    if not (s>0): return p
    p=p/s; S=float(-sum(float(v)*math.log(float(v)) for v in p if float(v)>0))
    if S<3.0:
        w=0.25+0.25*S; p=_np.power(p,w); s2=float(p.sum())
        p=p/s2 if s2>0 else p
    return p
def prep_query(mz,it,floor=0.002,topk=256):
    mz=_np.asarray(list(mz),dtype=_np.float64); it=_np.asarray(list(it),dtype=_np.float64)
    if len(mz)==0: return mz,it
    mx=float(it.max())
    if not (mx>0): return _np.zeros(0),_np.zeros(0)
    keep=it>=floor*mx; mz,it=mz[keep],it[keep]
    if len(mz)>topk:
        o=_np.argsort(it)[len(it)-topk:]; o=_np.sort(o); mz,it=mz[o],it[o]
    o=_np.argsort(mz); mz,it=mz[o],it[o]
    return mz,_entw(it).astype(_np.float64)
def entropy_sim(qmz,qp,cmz,cp,tol=TOL,ppm=PPM):
    qmz=_np.asarray(list(qmz),dtype=_np.float64); qp=_np.asarray(list(qp),dtype=_np.float64)
    cmz=_np.asarray(list(cmz),dtype=_np.float64); cp=_np.asarray(list(cp),dtype=_np.float64)
    n,m=len(qmz),len(cmz)
    if n==0 or m==0: return {'similarity':0.0,'n_matched':0,'explained_query':0.0}
    SA=float(-sum(float(v)*math.log(float(v)) for v in qp if float(v)>0))
    SB=float(-sum(float(v)*math.log(float(v)) for v in cp if float(v)>0))
    SAB=0.0; i=j=0; nm=0; mw=0.0; L4=math.log(4.0)
    while i<n and j<m:
        t=float(tol); pt=float(qmz[i])*float(ppm)*1e-6
        t=pt if pt>t else t; d=float(qmz[i])-float(cmz[j])
        if d<-t: v=float(qp[i])*0.5; i+=1
        elif d>t: v=float(cp[j])*0.5; j+=1
        else: v=(float(qp[i])+float(cp[j]))*0.5; nm+=1; mw+=float(qp[i]); i+=1; j+=1
        if v>0: SAB-=v*math.log(v)
    while i<n:
        v=float(qp[i])*0.5
        if v>0: SAB-=v*math.log(v)
        i+=1
    while j<m:
        v=float(cp[j])*0.5
        if v>0: SAB-=v*math.log(v)
        j+=1
    s=1.0-(2.0*SAB-SA-SB)/L4
    s=0.0 if s<1e-9 else (1.0 if s>1.0 else s)
    return {'similarity':float(s),'n_matched':int(nm),'explained_query':float(mw)}
def parse_mgf(path):
    out={}; cur=None; cur_peaks=None; cur_title=None
    def flush():
        if cur is not None and cur_peaks is not None:
            out[cur]={'mz':cur_peaks[0],'it':cur_peaks[1]}
    cur_key=None
    with open(path, errors='ignore') as f:
        for ln in f:
            s=ln.strip()
            if s=='BEGIN IONS': cur_peaks=([],[]); cur_title=None; continue
            if s=='END IONS':
                key=cur_title if cur_title else f'ORDER_{len(out)}'
                out[key]={'mz':cur_peaks[0],'it':cur_peaks[1]}; cur_peaks=None; cur_title=None; continue
            if cur_peaks is None: continue
            if '=' in s:
                k,v=s.split('=',1); k=k.strip().upper()
                if k in ('TITLE','NAME', 'SCANS') and not cur_title:
                    vv=v.strip()
                    if '__' in vv: cur_title=vv.split()[0]
                continue
            parts=s.split()
            if len(parts)>=2:
                try: cur_peaks[0].append(float(parts[0])); cur_peaks[1].append(float(parts[1]))
                except Exception: pass
    return out
pred={}
for tag in ['A','B']:
    d=parse_mgf(f'/kaggle/working/fiora_pred_{tag}.mgf')
    print(tag, 'parsed entries', len(d))
    pred.update(d)
rowsA=list(csv.DictReader(open('/kaggle/working/fiora_input_A.csv')))
rowsB=list(csv.DictReader(open('/kaggle/working/fiora_input_B.csv')))
rows=rowsA+rowsB
print('input rows', len(rows), 'pred entries', len(pred))
# 名称映射：优先 TITLE，缺失按顺序兜底（记录假设）
by_name={r['Name']:r for r in rows}
ordered_names=[r['Name'] for r in rows]
order_keys=[k for k in pred if k.startswith('ORDER_')]
print('titled hits', sum(1 for r in rows if r['Name'] in pred), 'order_fallback', len(order_keys))
if order_keys:
    unmapped=[r['Name'] for r in rows if r['Name'] not in pred]
    assert len(unmapped)==len(order_keys), f'map mismatch {len(unmapped)} vs {len(order_keys)}'
    for nm, ok in zip(unmapped, sorted(order_keys, key=lambda k: int(k.split('_')[1]))):
        pred[nm]=pred.pop(ok)
    print('order fallback mapped', len(unmapped))
import glob as _g
hits=sorted(_g.glob('/kaggle/input/**/np250_gl_input.json', recursive=True))
items=json.load(open(hits[0])); by_mid={x['mid']:x for x in items}
scores={}; per_mol={}
tot_scored=0; tot_fail=0
for mid in [r['mid'] for r in rows[:1]]:
    pass
mids_sorted=sorted({r['mid'] for r in rows})
for mid in mids_sorted:
    mol=by_mid[mid]; ad=by_name[[r['Name'] for r in rows if r['mid']==mid][0]]['Precursor_type']
    qspecs=[s for s in mol['spectra'] if s.get('adduct')==ad]
    qprep=[]
    for s in qspecs:
        try:
            m1,i1=clean_peaks(s['mz'],s['it'],float(s['prec']))
            m2,i2=clean_peaks(list(m1.astype(float)),list(i1.astype(float)),float(s['prec']))
            qm,qp=prep_query(list(m2.astype(float)),list(i2.astype(float)))
            if len(qm)>0: qprep.append((qm,qp))
        except Exception: pass
    rec={}; n_sc=0
    for r in [x for x in rows if x['mid']==mid]:
        smi=r['SMILES']; nm=r['Name']
        p=pred.get(nm)
        if p is None or len(p.get('mz',[]))==0:
            rec[smi]={'similarity':None,'n_matched':0,'explained_query':0.0,'status':'inference_failed','reason':'pred_missing'}; tot_fail+=1; continue
        try:
            pm,pp=prep_query(p['mz'],p['it'])
            if len(pm)==0:
                rec[smi]={'similarity':None,'n_matched':0,'explained_query':0.0,'status':'no_useful_peaks','reason':'empty_pred_after_prep'}; tot_fail+=1; continue
            sims=[]; nmt=0; ex=[]
            for qm,qp in qprep:
                d=entropy_sim(qm,qp,pm,pp)
                sims.append(d['similarity']); nmt+=d['n_matched']; ex.append(d['explained_query'])
            if not sims:
                rec[smi]={'similarity':None,'n_matched':0,'explained_query':0.0,'status':'no_useful_peaks','reason':'no_query'}; tot_fail+=1; continue
            ms=sum(sims)/len(sims)
            rec[smi]={'similarity':float(ms),'n_matched':int(nmt),'explained_query':float(sum(ex)/len(ex)) if ex else 0.0,'status':'scored','reason':'','n_spectra_scored':int(len(sims))}
            n_sc+=1; tot_scored+=1
        except Exception as e:
            rec[smi]={'similarity':None,'n_matched':0,'explained_query':0.0,'status':'inference_failed','reason':repr(e)[:120]}; tot_fail+=1
    scores[mid]=rec; per_mol[mid]={'n_cands':len(rec),'n_scored':n_sc,'n_failed':len(rec)-n_sc,'n_query_supported':len(qprep)}
    sims=[v['similarity'] for v in rec.values() if v['similarity'] is not None]
    per_mol[mid].update({'mean_sim':float(sum(sims)/len(sims)) if sims else 0.0,'max_sim':float(max(sims)) if sims else 0.0})
    print(mid, per_mol[mid])
assert tot_scored>0, 'all predictions failed'
try:
    import torch
    vram = float(torch.cuda.max_memory_allocated()/2**30) if torch.cuda.is_available() else 0.0
except Exception: vram = 0.0
summary={'model':'fiora-OS-v1.0.0-msnlib-v7','commit':'e19ef82c9a6cb9dbac92bce23e914008f1aeb44e','weight_bytes':53845076,'instrument':'HCD(timsTOF->HCD assumption)','ce':'eV meanCE (no NCE)','n_molecules':len(per_mol),'n_cands':int(sum(v['n_cands'] for v in per_mol.values())),'n_scored':int(tot_scored),'n_failed':int(tot_fail),'seconds_total':round(time.time()-t0,1),'max_mem_gb':vram,'per_molecule':per_mol,'scoring':'entropy TOL0.01 PPM20 (GL-aligned choice, not FIORA native)','offline_note':'53MB weight + pinned source can be bundled as dataset for offline; this DEV used internet only for download+pip'}
print(json.dumps({k:summary[k] for k in ['n_molecules','n_cands','n_scored','n_failed','seconds_total','max_mem_gb']},indent=1))
json.dump(scores, open('/kaggle/working/fiora_scores_22.json','w'))
json.dump(summary, open('/kaggle/working/fiora_pilot_summary.json','w'), indent=1)
print('wrote /kaggle/working/fiora_scores_22.json + fiora_pilot_summary.json in', round(time.time()-t0,1), 's')
